# Solina — Text-to-Speech for Windows

**Solina** is a minimalist dark-themed Windows desktop app that turns text into MP3 audiobooks.
It supports importing `.txt`, `.pdf`, and `.epub` files, splitting the content with several
strategies, previewing voices live, and exporting sequentially named MP3 files.

## Requirements
- **64-bit Windows 10/11**
- **64-bit Python 3.10+**
- **ffmpeg** on `PATH` (required by `pydub` for MP3 export)

## Python dependencies
| Package | Purpose |
|---|---|
| `pyttsx3` | Cross-platform TTS wrapper (uses Windows SAPI5) |
| `comtypes` | COM bindings used by `pyttsx3` on Windows |
| `pydub` | WAV → MP3 conversion |
| `PyMuPDF` (`fitz`) | PDF text extraction |
| `ebooklib` | EPUB parsing |
| `beautifulsoup4` | Strips HTML from EPUB chapters |
| `ttkbootstrap` | Modern dark theme for Tkinter |

## ffmpeg setup (needed for MP3 export)
`pydub` shells out to `ffmpeg` to encode MP3. Without it, export will fail.

**Option A — winget (recommended):**
```powershell
winget install Gyan.FFmpeg

In [8]:

# Install all required packages into the current kernel's interpreter.
# Using sys.executable guarantees we install into the SAME Python that Jupyter is running.
import sys, subprocess

PACKAGES = [
    "pyttsx3",
    "comtypes",
    "pydub",
    "PyMuPDF",
    "ebooklib",
    "beautifulsoup4",
    "ttkbootstrap",
]

# --quiet keeps the notebook output clean; remove it if you want to see progress bars.
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", *PACKAGES])
print("All Solina dependencies installed.")

All Solina dependencies installed.


### Cell 3 — Imports + 64-bit guard  **(REPLACES the existing imports cell)**

**What it does:** Adds the pieces the fixed conversion pipeline needs:
- `pythoncom` — COM apartment init/teardown for worker threads.
- `comtypes.client` — direct SAPI5 `SpVoice` / `SpFileStream` access.
- `shutil` — to locate `ffmpeg`.
- `traceback` — to write full stack traces to `solina_log.txt`.
- `subprocess` — used by the diagnostic cell.
- A tiny `log_exc()` helper that appends tracebacks to `solina_log.txt`
  next to the script (or in the notebook's working directory).

**Why:** The old `pyttsx3.save_to_file()` path silently produces no file for
many voices (especially `Microsoft * Online` voices) and swallows errors.
The new pipeline uses raw SAPI5 via comtypes and logs everything.

**Goes after:** the app-overview markdown cell (cell 1) — same position as the
current imports cell. **Replaces it entirely.**

In [1]:
# ---------------------------------------------------------------------------
# Standard library
# ---------------------------------------------------------------------------
import sys
import os
import re
import time
import queue
import shutil
import platform
import tempfile
import threading
import traceback
import subprocess
from pathlib import Path
from tkinter import filedialog, messagebox

# ---------------------------------------------------------------------------
# 64-bit / Windows guard
# ---------------------------------------------------------------------------
if sys.maxsize <= 2 ** 32:
    raise SystemExit(
        "Solina requires a 64-bit Python build.\n"
        f"Detected: {platform.architecture()[0]} on {platform.machine()}.\n"
        "Install 64-bit Python 3.10+ from python.org and re-run the notebook."
    )

if platform.system() != "Windows":
    # We don't hard-fail so the notebook can still be reviewed on other OSes,
    # but SAPI5 voice enumeration will not work outside Windows.
    print("WARNING: Solina targets Windows. Voice enumeration may fail here.")

print(f"Environment OK — Python {sys.version.split()[0]} ({platform.architecture()[0]})")

# ---------------------------------------------------------------------------
# GUI / TTS / parsing third-party packages
# ---------------------------------------------------------------------------
import tkinter as tk
import ttkbootstrap as tb
from ttkbootstrap.constants import (
    BOTH, X, Y, LEFT, RIGHT, W, E, N, S, END, HORIZONTAL,
)

import pyttsx3

import fitz  # PyMuPDF
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup

from pydub import AudioSegment

# ---------------------------------------------------------------------------
# COM / SAPI5 — direct access used for file export (preview still uses pyttsx3)
# ---------------------------------------------------------------------------
import pythoncom
import comtypes.client

# ---------------------------------------------------------------------------
# Logging helper — writes full tracebacks to solina_log.txt
# ---------------------------------------------------------------------------
try:
    _SCRIPT_DIR = Path(__file__).resolve().parent     # when run as solina.py
except NameError:
    _SCRIPT_DIR = Path.cwd()                          # when run inside Jupyter

LOG_PATH = _SCRIPT_DIR / "solina_log.txt"


def log_exc(context: str, exc: BaseException) -> str:
    """
    Append the full traceback of `exc` to solina_log.txt, tagged with `context`.

    Returns a short one-line summary suitable for showing in the UI.
    Never raises — logging must not break the export loop.
    """
    tb_str = "".join(
        traceback.format_exception(type(exc), exc, exc.__traceback__)
    )
    stamp = time.strftime("%Y-%m-%d %H:%M:%S")
    try:
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(f"\n[{stamp}] {context}\n{tb_str}\n")
    except Exception:
        # If even logging fails, we still return a summary — don't crash.
        pass
    return f"{context}: {type(exc).__name__}: {exc}"


print(f"Log file: {LOG_PATH}")

Environment OK — Python 3.11.15 (64bit)
Log file: c:\Users\Tarun\Documents\SOLINA\solina_log.txt


### Cell 6 — `TTSEngine`  **(REPLACES the existing TTS-engine cell)**

**What it does:**
- Keeps `preview()` on **pyttsx3** (your Test-voice button already works).
- Rewrites `render_to_wav()` to use **raw SAPI5 COM objects**
  (`SAPI.SpVoice` + `SAPI.SpFileStream`) instead of `pyttsx3.save_to_file()`.
- Sets rate as SAPI's native `-10..10` (mapped from your WPM slider),
  volume as SAPI `0..100`, and applies **pitch** through SSML XML
  (`<pitch absmiddle="N"/>`) because SAPI has no direct pitch property.
- Picks the voice by its token **Id** (the same Id pyttsx3 exposes in
  `voice.id`), so the GUI dropdown keeps working unchanged.

**Why:** `pyttsx3.save_to_file()` reuses an internal stream and silently
fails on the 2nd+ chunk or with `Microsoft * Online` voices. Raw SAPI5 with
a fresh `SpVoice` + `SpFileStream` per chunk — and an explicit `stream.Close()`
— is reliable. Caller is responsible for `pythoncom.CoInitialize()` in the
thread that invokes this (the worker does it).

**Goes after:** the text-splitting cell (cell 5). **Replaces the TTSEngine cell.**

In [14]:
# ---- TTS engine: preview via pyttsx3, file export via direct SAPI5 (comtypes) ----
from pathlib import Path
import pyttsx3
import comtypes.client


class TTSEngine:
    """
    Wraps pyttsx3/SAPI5.
    - preview() uses pyttsx3 (Test voice button).
    - render_to_wav() uses direct SAPI5/COM so file export is reliable.
    """

    def __init__(self):
        # pyttsx3 engine is only used for preview + voice enumeration
        self.engine = pyttsx3.init(driverName="sapi5")
        self.voices = self.engine.getProperty("voices") or []

    def list_voices(self) -> list[dict]:
        """Return installed SAPI5 voices as dicts: {id, name, langs}."""
        out = []
        for v in self.voices:
            langs = []
            for lang in (getattr(v, "languages", None) or []):
                if isinstance(lang, (bytes, bytearray)):
                    lang = lang.decode("utf-8", "ignore")
                langs.append(str(lang))
            out.append({"id": v.id, "name": v.name, "langs": langs})
        return out

    @staticmethod
    def _apply(engine, voice_id, rate, pitch, volume):
        """Apply settings to a pyttsx3 engine (preview path only)."""
        engine.setProperty("rate", int(rate))
        engine.setProperty("volume", max(0.0, min(1.0, float(volume))))
        if voice_id:
            engine.setProperty("voice", voice_id)
        # pitch is not supported by pyttsx3's SAPI5 driver, so it is skipped here

    def preview(self, text: str, voice_id: str, rate: int, pitch: int, volume: float):
        """Speak a short sample through the speakers (no file written)."""
        engine = pyttsx3.init(driverName="sapi5")
        self._apply(engine, voice_id, rate, pitch, volume)
        engine.say(text)
        engine.runAndWait()
        try:
            engine.stop()
        except Exception:
            pass

    def render_to_wav(self, text: str, wav_path: Path,
                      voice_id: str, rate: int, pitch: int, volume: float):
        """
        Render text to a WAV file using direct SAPI5 (SpVoice + SpFileStream).
        The calling thread must have called pythoncom.CoInitialize().
        Rate: WPM (50..400, default 180) -> SAPI -10..10.
        Pitch: -50..50 -> SAPI -10..10 via <pitch absmiddle="N"/> XML.
        Volume: 0.0..1.0 -> 0..100.
        """
        voice = comtypes.client.CreateObject("SAPI.SpVoice")
        stream = comtypes.client.CreateObject("SAPI.SpFileStream")

        # Select the requested voice by token Id
        if voice_id:
            tokens = voice.GetVoices()
            for i in range(tokens.Count):
                tok = tokens.Item(i)
                if tok.Id == voice_id:
                    voice.Voice = tok
                    break

        voice.Rate = max(-10, min(10, int(round((int(rate) - 180) / 20.0))))
        voice.Volume = int(max(0.0, min(1.0, float(volume))) * 100)

        wav_path = Path(wav_path)
        stream.Open(str(wav_path), 3)          # 3 = SSFMCreateForWrite
        voice.AudioOutputStream = stream

        try:
            if int(pitch) != 0:
                sapi_pitch = max(-10, min(10, int(round(int(pitch) / 5.0))))
                safe = (text.replace("&", "&amp;")
                            .replace("<", "&lt;")
                            .replace(">", "&gt;"))
                voice.Speak(f'<pitch absmiddle="{sapi_pitch}"/>{safe}', 8)  # 8 = SVSFIsXML
            else:
                voice.Speak(text)              # synchronous: returns when the WAV is written
        finally:
            try:
                stream.Close()                 # finalizes the WAV header on disk
            except Exception:
                pass

### Cell 7 — `wav_to_mp3` + ffmpeg detection  **(REPLACES the existing wav_to_mp3 cell)**

**What it does:**
- Adds `_find_ffmpeg()`, which checks `PATH` (via `shutil.which`) and then the
  directory of `solina.py` (or the notebook CWD) for `ffmpeg.exe`.
- If found next to the script, sets `AudioSegment.converter` (and `ffprobe`)
  so pydub uses it even when `ffmpeg` isn't on the system PATH.
- Rewrites `wav_to_mp3()` to (a) raise a clear `RuntimeError` when ffmpeg is
  missing, (b) verify the MP3 exists and is `> 0` bytes, (c) delete the temp
  WAV after a successful conversion.

**Why:** Your symptom (no files anywhere, no error) is consistent with pydub
silently no-oping when ffmpeg is missing. Explicit detection + verification
turns that into a visible, actionable error.

**Goes after:** the TTSEngine cell (cell 6). **Replaces the wav_to_mp3 cell.**

In [5]:
import shutil
from pathlib import Path
from pydub import AudioSegment


def _find_ffmpeg() -> str | None:
    """
    Return the path to an ffmpeg executable, or None if not found.

    Search order:
      1. System PATH              (shutil.which)
      2. Directory of solina.py   (or notebook CWD if __file__ is undefined)
    """
    found = shutil.which("ffmpeg")
    if found:
        return found

    for name in ("ffmpeg.exe", "ffmpeg"):
        candidate = _SCRIPT_DIR / name
        if candidate.is_file():
            return str(candidate)
    return None


# Try to configure pydub with a locally-bundled ffmpeg, if one exists.
_FFMPEG = _find_ffmpeg()
if _FFMPEG:
    AudioSegment.converter = _FFMPEG
    # Also point pydub at ffprobe next to it, if present.
    _ffprobe_name = "ffprobe.exe" if _FFMPEG.lower().endswith(".exe") else "ffprobe"
    _ffprobe = Path(_FFMPEG).with_name(_ffprobe_name)
    if _ffprobe.is_file():
        AudioSegment.ffprobe = str(_ffprobe)


def wav_to_mp3(wav_path: Path, mp3_path: Path) -> None:
    """
    Convert a WAV file to MP3 using pydub/ffmpeg, then delete the WAV.

    Raises RuntimeError with an actionable message if ffmpeg is missing or
    if the resulting MP3 is absent / empty.
    """
    if _find_ffmpeg() is None:
        raise RuntimeError(
            "ffmpeg was not found.\n"
            "Install it with:  winget install Gyan.FFmpeg\n"
            "Then RESTART Jupyter / your terminal so PATH is refreshed.\n"
            "Or drop ffmpeg.exe next to solina.py."
        )

    wav_path = Path(wav_path)
    mp3_path = Path(mp3_path)

    # Read the WAV — surface a clear error if SAPI wrote something unreadable.
    try:
        audio = AudioSegment.from_wav(str(wav_path))
    except Exception as e:
        raise RuntimeError(f"Could not read WAV {wav_path.name}: {e}") from e

    # Encode to MP3.
    try:
        audio.export(str(mp3_path), format="mp3")
    except FileNotFoundError as e:
        raise RuntimeError(
            "ffmpeg was not found when encoding MP3. Install it and restart."
        ) from e

    # Verify the output — no silent zero-byte files.
    if not mp3_path.exists() or mp3_path.stat().st_size == 0:
        raise RuntimeError(f"MP3 was not written or is empty: {mp3_path}")

    # Clean up the temp WAV.
    try:
        wav_path.unlink()
    except Exception:
        pass

### NEW — Standalone diagnostic  **(NEW cell, insert AFTER the wav_to_mp3 cell, BEFORE the SolinaApp cell)**

**What it does:** Runs a step-by-step end-to-end sanity check and prints
PASS / FAIL for each stage:
1. 64-bit Python
2. Running on Windows
3. ffmpeg located
4. Output folder writable
5. SAPI voices enumerated
6. One-sentence WAV export via direct SAPI5
7. WAV → MP3 conversion via pydub

**Why:** When something silently fails, run this cell. It tells you exactly
which stage is broken. Any exception is also written to `solina_log.txt`.

**Goes after:** the wav_to_mp3 cell (cell 7). The SolinaApp cell (old cell 8)
becomes the next cell.

In [15]:
import sys
import os
import time
import shutil
import platform
import logging
import traceback
from pathlib import Path

import pythoncom            # COM init for worker threads
import comtypes.client      # direct SAPI5 access
import pyttsx3
from pydub import AudioSegment



def run_diagnostics() -> list[tuple[str, bool, str]]:
    """
    Step-by-step PASS/FAIL diagnostic. Prints a report and returns a list of
    (name, ok, detail) tuples. Safe to call as often as you like.
    """
    results: list[tuple[str, bool, str]] = []

    def record(name: str, ok: bool, detail: str = "") -> None:
        results.append((name, ok, detail))
        flag = "PASS" if ok else "FAIL"
        suffix = f" — {detail}" if detail else ""
        print(f"[{flag}] {name}{suffix}")

    print("=" * 60)
    print(" Solina diagnostics")
    print("=" * 60)

    # 1) 64-bit Python -----------------------------------------------------
    record(
        "64-bit Python",
        sys.maxsize > 2 ** 32,
        f"{platform.architecture()[0]}, Python {sys.version.split()[0]}",
    )

    # 2) Windows -----------------------------------------------------------
    record("Running on Windows", platform.system() == "Windows", platform.system())

    # 3) ffmpeg ------------------------------------------------------------
    ff = _find_ffmpeg()
    record(
        "ffmpeg located",
        ff is not None,
        ff or "not found — install with: winget install Gyan.FFmpeg",
    )

    # 4) Output folder writable -------------------------------------------
    out = Path.home() / "Solina_Diag"
    try:
        out.mkdir(parents=True, exist_ok=True)
        probe = out / ".write_test"
        probe.write_text("ok", encoding="utf-8")
        probe.unlink()
        record("Output folder writable", True, str(out))
    except Exception as e:
        record("Output folder writable", False, str(e))

    # 5) SAPI voices -------------------------------------------------------
    try:
        tts = TTSEngine()
        voices = tts.list_voices()
        record("SAPI voices enumerated", len(voices) > 0, f"{len(voices)} voice(s)")
        for v in voices[:8]:
            print(f"       - {v['name']}")
        if not voices:
            print("       (no voices found — Windows SAPI may be misconfigured)")
            return results
    except Exception as e:
        log_exc("diagnostic: enumerate voices", e)
        record("SAPI voices enumerated", False, str(e))
        return results

    # 6) WAV export via direct SAPI5 ---------------------------------------
    wav_path = out / "diag.wav"
    wav_ok = False
    pythoncom.CoInitialize()
    try:
        tts.render_to_wav(
            "Diagnostic test sentence.",
            wav_path,
            voice_id=voices[0]["id"],
            rate=180, pitch=0, volume=1.0,
        )
        wav_ok = wav_path.exists() and wav_path.stat().st_size > 0
        size = wav_path.stat().st_size if wav_path.exists() else 0
        record("WAV export (direct SAPI5)", wav_ok, f"{size} bytes  ({voices[0]['name']})")
    except Exception as e:
        log_exc("diagnostic: WAV export", e)
        record("WAV export (direct SAPI5)", False, str(e))
    finally:
        pythoncom.CoUninitialize()

    # 7) WAV → MP3 ---------------------------------------------------------
    if wav_ok:
        try:
            mp3_path = out / "diag.mp3"
            wav_to_mp3(wav_path, mp3_path)
            mp3_ok = mp3_path.exists() and mp3_path.stat().st_size > 0
            size = mp3_path.stat().st_size if mp3_path.exists() else 0
            record("WAV -> MP3 conversion", mp3_ok, f"{size} bytes")
        except Exception as e:
            log_exc("diagnostic: MP3 conversion", e)
            record("WAV -> MP3 conversion", False, str(e))
    else:
        record("WAV -> MP3 conversion", False, "skipped (no WAV produced)")

    print("=" * 60)
    passed = sum(1 for _, ok, _ in results if ok)
    print(f" Summary: {passed}/{len(results)} checks passed")
    print(f" Log file: {LOG_PATH}")
    print("=" * 60)
    return results


# Run the diagnostic immediately when this cell executes.
run_diagnostics()

 Solina diagnostics
[PASS] 64-bit Python — 64bit, Python 3.11.15
[PASS] Running on Windows — Windows
[PASS] ffmpeg located — C:\ffmpeg-2026-08-03-git-01a25f74cc-essentials_build\bin\ffmpeg.EXE
[PASS] Output folder writable — C:\Users\Tarun\Solina_Diag
[PASS] SAPI voices enumerated — 40 voice(s)
       - Microsoft David Desktop - English (United States)
       - Microsoft Hazel Desktop - English (Great Britain)
       - Microsoft Amber Native - English (United States)
       - Microsoft Andrew Native - English (United States)
       - Microsoft Aria Native - English (United States)
       - Microsoft Ava Native - English (United States)
       - Microsoft Christopher Native - English (United States)
       - Microsoft Hazel - English (United Kingdom)
[PASS] WAV export (direct SAPI5) — 111136 bytes  (Microsoft David Desktop - English (United States))
[PASS] WAV -> MP3 conversion — 10570 bytes
 Summary: 7/7 checks passed


NameError: name 'LOG_PATH' is not defined

In [16]:
# ---- SETUP CELL: shared imports + logging helpers (run this FIRST) ----
import sys, os, time, shutil, platform, logging, traceback
from pathlib import Path

import pythoncom            # COM init for worker threads
import comtypes.client      # direct SAPI5 access
import pyttsx3
from pydub import AudioSegment

# Folder of solina.py; falls back to the notebook's working folder in Jupyter
try:
    _SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    _SCRIPT_DIR = Path.cwd()

# Every error goes to solina_log.txt next to the script
LOG_PATH = _SCRIPT_DIR / "solina_log.txt"
logging.basicConfig(
    filename=str(LOG_PATH),
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    force=True,
)

def log_exc(context: str, exc: Exception) -> None:
    """Write a full traceback for `exc` to solina_log.txt."""
    logging.error("%s: %s\n%s", context, exc, traceback.format_exc())

### Cell 8 — `SolinaApp`  **(REPLACES the existing GUI cell)**

**What it does:**
- `_on_convert` now performs strict validation *before* launching the worker:
  non-empty text, ≥ 1 chunk, a voice selected, and **ffmpeg present** — each
  failure produces a clear messagebox instead of a silent no-op.
- `_convert_worker` runs the export in a background thread that:
  - calls `pythoncom.CoInitialize()` on entry and `CoUninitialize()` in
    `finally` (fixes the "no audio, no error" COM-apartment bug),
  - wraps every chunk in `try/except` that logs the full traceback via
    `log_exc()` and appends a hint when the failing voice is an
    `Online`/`Natural` voice (these often can't render to file),
  - skips empty chunks, verifies each MP3 exists and is `> 0` bytes,
  - updates the UI **only through `self.after(0, ...)`** so nothing from the
    worker thread touches Tk directly.
- On finish, a custom dialog shows the count + folder with an
  **Open folder** button (`os.startfile`).
- `_drain_queue` / `_msg_q` are removed — no longer needed.

**Why:** This is the actual bug fix. Preview works because pyttsx3 runs on the
Tk main thread (COM already initialised). Export silently produced nothing
because the worker thread had no COM apartment and `save_to_file()` was
reusing a stale stream. Both are now fixed.

**Goes after:** the diagnostic cell. **Replaces the SolinaApp cell.**

In [17]:
# Palette (unchanged from the original app)
ACCENT   = "#4ea1ff"
BG_MAIN  = "#2b2b2b"
BG_DARK  = "#1e1e1e"
FG_TEXT  = "#e6e6e6"


class SolinaApp(tb.Window):
    """Solina main window."""

    # --------------------------------------------------------------- lifecycle
    def __init__(self):
        super().__init__(themename="darkly")
        self.title("Solina — Text to Speech")
        self.geometry("1180x800")
        self.minsize(960, 640)
        self.configure(background=BG_MAIN)

        # Core objects ------------------------------------------------------
        self.tts = TTSEngine()
        self._busy = False

        # Tk variables ------------------------------------------------------
        self.voice_choice = tk.StringVar()
        self.rate_var     = tk.IntVar(value=180)
        self.pitch_var    = tk.IntVar(value=0)
        self.volume_var   = tk.DoubleVar(value=1.0)

        self.split_mode   = tk.StringVar(value="chars")
        self.char_count   = tk.StringVar(value="3000")
        self.word_count   = tk.StringVar(value="500")
        self.keywords     = tk.StringVar(value="")
        self.strip_kw     = tk.BooleanVar(value=True)

        self.output_dir   = tk.StringVar(value=str(Path.home() / "Solina_Output"))
        self.status_var   = tk.StringVar(value="Ready")
        self.file_var     = tk.StringVar(value="(no file loaded)")

        self._voice_map: dict[str, str] = {}   # display label -> SAPI voice id

        # Build & populate --------------------------------------------------
        self._build_ui()
        self._populate_voices()
        self.protocol("WM_DELETE_WINDOW", self._on_close)

    # ------------------------------------------------------------------ build
    def _build_ui(self):
        """Create every widget in the window exactly once."""
        style = self.style
        style.configure("Solina.TFrame", background=BG_MAIN)
        style.configure("Solina.TLabel", background=BG_MAIN, foreground=FG_TEXT)
        style.configure("Solina.TLabelframe", background=BG_MAIN,
                        foreground=FG_TEXT, bordercolor="#3c3c3c")
        style.configure("Solina.TLabelframe.Label", background=BG_MAIN, foreground=FG_TEXT)
        style.configure("Solina.TRadiobutton", background=BG_MAIN, foreground=FG_TEXT)
        style.configure("Solina.TCheckbutton", background=BG_MAIN, foreground=FG_TEXT)
        style.configure("Solina.TEntry", fieldbackground=BG_DARK, foreground=FG_TEXT)
        style.configure("Accent.TButton", background=ACCENT, foreground="#0b0b0b")

        root = tb.Frame(self, padding=14, style="Solina.TFrame")
        root.pack(fill=BOTH, expand=True)
        root.columnconfigure(0, weight=1)
        root.columnconfigure(1, weight=0)
        root.rowconfigure(1, weight=1)

        # ---- Row 0: toolbar ------------------------------------------------
        bar = tb.Frame(root, style="Solina.TFrame")
        bar.grid(row=0, column=0, columnspan=2, sticky="ew", pady=(0, 12))

        tb.Button(bar, text="Import file…", bootstyle="secondary",
                  command=self._on_import).pack(side=LEFT)
        tb.Label(bar, textvariable=self.file_var, style="Solina.TLabel"
                 ).pack(side=LEFT, padx=(10, 0))

        tb.Button(bar, text="Browse…", bootstyle="secondary",
                  command=self._on_browse_output).pack(side=RIGHT)
        tb.Label(bar, textvariable=self.output_dir, style="Solina.TLabel"
                 ).pack(side=RIGHT, padx=(0, 10))
        tb.Label(bar, text="Output folder:", style="Solina.TLabel").pack(side=RIGHT)

        # ---- Row 1 left: text editor ---------------------------------------
        left = tb.Frame(root, style="Solina.TFrame")
        left.grid(row=1, column=0, sticky="nsew", padx=(0, 14))
        left.rowconfigure(1, weight=1)
        left.columnconfigure(0, weight=1)

        tb.Label(left, text="Text content", style="Solina.TLabel",
                 font=("Segoe UI", 12, "bold")).grid(row=0, column=0, sticky=W)

        text_wrap = tk.Frame(left, bg=BG_DARK)
        text_wrap.grid(row=1, column=0, sticky="nsew", pady=(6, 0))
        text_wrap.rowconfigure(0, weight=1)
        text_wrap.columnconfigure(0, weight=1)

        self.text = tk.Text(
            text_wrap, wrap="word", undo=True,
            bg=BG_DARK, fg=FG_TEXT, insertbackground=FG_TEXT,
            selectbackground=ACCENT, selectforeground="#000000",
            relief="flat", padx=12, pady=12, font=("Consolas", 11),
        )
        self.text.grid(row=0, column=0, sticky="nsew")

        sb = tb.Scrollbar(text_wrap, orient="vertical", command=self.text.yview)
        sb.grid(row=0, column=1, sticky="ns")
        self.text.configure(yscrollcommand=sb.set)

        # ---- Row 1 right: controls -----------------------------------------
        right = tb.Frame(root, style="Solina.TFrame")
        right.grid(row=1, column=1, sticky="ns")

        # Voice + audio
        audio_lf = tb.Labelframe(right, text="Voice & audio", padding=12,
                                 style="Solina.TLabelframe")
        audio_lf.pack(fill=X, pady=(0, 10))

        tb.Label(audio_lf, text="Voice", style="Solina.TLabel").pack(anchor=W)
        self.voice_combo = tb.Combobox(audio_lf, textvariable=self.voice_choice,
                                       state="readonly", width=38)
        self.voice_combo.pack(fill=X, pady=(2, 10))

        self._slider(audio_lf, "Rate",   self.rate_var,   50, 400, 1)
        self._slider(audio_lf, "Pitch",  self.pitch_var, -50,  50, 1)
        self._slider(audio_lf, "Volume", self.volume_var,  0.0, 1.0, 0.01)

        tb.Button(audio_lf, text="Test voice", bootstyle="info",
                  command=self._on_test_voice).pack(fill=X, pady=(10, 0))

        # Splitting
        split_lf = tb.Labelframe(right, text="Text splitting", padding=12,
                                 style="Solina.TLabelframe")
        split_lf.pack(fill=X)

        def radio(value, label):
            tb.Radiobutton(split_lf, text=label, value=value,
                           variable=self.split_mode,
                           style="Solina.TRadiobutton").pack(anchor=W)

        radio("chars",    "By block size — characters")
        row = tb.Frame(split_lf, style="Solina.TFrame"); row.pack(fill=X, padx=(22, 0))
        tb.Entry(row, textvariable=self.char_count, width=8).pack(side=LEFT)
        tb.Label(row, text="characters", style="Solina.TLabel").pack(side=LEFT, padx=6)

        radio("words",    "By block size — words")
        row = tb.Frame(split_lf, style="Solina.TFrame"); row.pack(fill=X, padx=(22, 0))
        tb.Entry(row, textvariable=self.word_count, width=8).pack(side=LEFT)
        tb.Label(row, text="words", style="Solina.TLabel").pack(side=LEFT, padx=6)

        radio("keywords", "By keyword list")
        row = tb.Frame(split_lf, style="Solina.TFrame"); row.pack(fill=X, padx=(22, 0))
        tb.Entry(row, textvariable=self.keywords, width=26).pack(side=LEFT)
        tb.Checkbutton(split_lf, text="Strip keyword from output",
                       variable=self.strip_kw,
                       style="Solina.TCheckbutton").pack(anchor=W, padx=(22, 0))

        radio("blank",    "By two consecutive empty lines")
        radio("caps",     "By lines that are ALL CAPS (headings)")

        # Export
        tb.Button(right, text="Convert to MP3", bootstyle="success",
                  command=self._on_convert).pack(fill=X, pady=(12, 0))

        # ---- Row 2: status + progress --------------------------------------
        bottom = tb.Frame(root, style="Solina.TFrame")
        bottom.grid(row=2, column=0, columnspan=2, sticky="ew", pady=(12, 0))
        bottom.columnconfigure(0, weight=1)

        self.progress = tb.Progressbar(bottom, mode="determinate",
                                       bootstyle="info-striped")
        self.progress.grid(row=0, column=0, sticky="ew")
        tb.Label(bottom, textvariable=self.status_var, style="Solina.TLabel"
                 ).grid(row=1, column=0, sticky=W, pady=(6, 0))

    def _slider(self, parent, label, var, lo, hi, step):
        """Helper: build a labelled horizontal slider."""
        row = tb.Frame(parent, style="Solina.TFrame")
        row.pack(fill=X, pady=(2, 0))
        tb.Label(row, text=label, width=7, style="Solina.TLabel").pack(side=LEFT)
        tb.Scale(row, from_=lo, to=hi, variable=var, orient=HORIZONTAL
                 ).pack(side=LEFT, fill=X, expand=True)

    # ------------------------------------------------------------------ voices
    def _populate_voices(self):
        """Fill the voice dropdown with installed SAPI5 voices."""
        voices = self.tts.list_voices()
        self._voice_map.clear()
        labels = []
        for v in voices:
            langs = ", ".join(v["langs"]) if v["langs"] else "unknown"
            label = f'{v["name"]}  ({langs})'
            labels.append(label)
            self._voice_map[label] = v["id"]
        self.voice_combo["values"] = labels
        if labels:
            self.voice_combo.current(0)

    # --------------------------------------------------------------- callbacks
    def _on_import(self):
        """Load .txt/.pdf/.epub into the text box."""
        path = filedialog.askopenfilename(
            title="Choose a source file",
            filetypes=[("Supported files", "*.txt *.pdf *.epub"),
                       ("Text", "*.txt"), ("PDF", "*.pdf"), ("EPUB", "*.epub")],
        )
        if not path:
            return
        try:
            content = extract_text(path)
        except Exception as e:
            log_exc("import", e)
            messagebox.showerror("Solina", f"Failed to read file:\n{e}")
            return
        self.text.delete("1.0", END)
        self.text.insert("1.0", content)
        self.file_var.set(Path(path).name)

    def _on_browse_output(self):
        """Pick the output folder for exported MP3s."""
        folder = filedialog.askdirectory(title="Choose output folder")
        if folder:
            self.output_dir.set(folder)

    def _on_test_voice(self):
        """Speak a sample using the current settings (no file export)."""
        if self._busy:
            return
        voice_id = self._voice_map.get(self.voice_choice.get())
        rate, pitch = self.rate_var.get(), self.pitch_var.get()
        vol = self.volume_var.get()
        sample = "Hello, this is Solina. This is how your audiobook will sound."
        self.status_var.set("Previewing voice…")

        def worker():
            try:
                self.tts.preview(sample, voice_id, rate, pitch, vol)
                self.after(0, self.status_var.set, "Preview finished.")
            except Exception as e:
                log_exc("preview", e)
                self.after(0, self.status_var.set, f"Preview failed: {e}")

        threading.Thread(target=worker, daemon=True).start()

    def _make_chunks(self, text: str) -> list[str]:
        """Apply the currently selected splitting strategy."""
        mode = self.split_mode.get()
        if mode == "chars":
            return split_by_chars(text, int(self.char_count.get() or 3000))
        if mode == "words":
            return split_by_words(text, int(self.word_count.get() or 500))
        if mode == "keywords":
            return split_by_keywords(text, self.keywords.get().split(","),
                                     strip=self.strip_kw.get())
        if mode == "blank":
            return split_by_blank_lines(text)
        if mode == "caps":
            return split_by_capital_lines(text)
        return [text]

    # ---------------------------------------------------------------- convert
    def _on_convert(self):
        """Validate inputs, confirm the file count, and start the export thread."""
        if self._busy:
            return

        # --- 1) text is not empty -------------------------------------------
        text = self.text.get("1.0", END).strip()
        if not text:
            messagebox.showwarning("Solina", "There is no text to convert.")
            return

        # --- 2) a voice is selected -----------------------------------------
        voice_label = self.voice_choice.get()
        voice_id = self._voice_map.get(voice_label)
        if not voice_id:
            messagebox.showwarning("Solina", "Please choose a voice from the dropdown.")
            return

        # --- 3) ffmpeg present ----------------------------------------------
        if _find_ffmpeg() is None:
            messagebox.showerror(
                "Solina — ffmpeg missing",
                "ffmpeg was not found. MP3 export needs it.\n\n"
                "Install with:  winget install Gyan.FFmpeg\n"
                "Then restart Jupyter / your terminal.\n\n"
                "Alternatively, place ffmpeg.exe next to solina.py."
            )
            return

        # --- 4) split gives at least 1 chunk --------------------------------
        try:
            chunks = self._make_chunks(text)
        except ValueError as e:
            messagebox.showerror("Solina", f"Invalid split parameter: {e}")
            return

        # Drop empties up front — the worker skips them too, but this keeps
        # the file-count preview honest.
        chunks = [c for c in chunks if c and c.strip()]
        if not chunks:
            messagebox.showwarning("Solina", "The split settings produced 0 chunks.")
            return

        # --- 5) confirm approximate file count ------------------------------
        proceed = messagebox.askyesno(
            "Solina",
            f"Approximate number of files: {len(chunks)}\n\nProceed with export?",
        )
        if not proceed:
            return

        # --- 6) ensure output folder exists ---------------------------------
        out_dir = Path(self.output_dir.get()).expanduser()
        try:
            os.makedirs(out_dir, exist_ok=True)
        except Exception as e:
            log_exc("create output folder", e)
            messagebox.showerror("Solina", f"Could not create output folder:\n{e}")
            return

        # --- 7) snapshot settings (workers must not read Tk state) ----------
        settings = {
            "voice_id":    voice_id,
            "voice_label": voice_label,
            "rate":        self.rate_var.get(),
            "pitch":       self.pitch_var.get(),
            "volume":      self.volume_var.get(),
        }

        # --- 8) launch worker -----------------------------------------------
        self._busy = True
        self.progress.configure(maximum=len(chunks), value=0)
        self.status_var.set(f"0 / {len(chunks)}")

        threading.Thread(
            target=self._convert_worker,
            args=(chunks, out_dir, settings),
            daemon=True,
        ).start()

    # ----------------------------------------------------------------- worker
    def _convert_worker(self, chunks, out_dir: Path, settings: dict):
        """
        Runs on a background thread. Owns its own COM apartment via
        CoInitialize/CoUninitialize. All UI updates go through self.after(0, …).
        """
        # ---- COM apartment for this thread (fixes silent "no output") ------
        pythoncom.CoInitialize()

        tmp_dir = Path(tempfile.mkdtemp(prefix="solina_"))
        ok = 0
        errors: list[str] = []
        voice_label = settings.get("voice_label", "")
        voice_id = settings.get("voice_id")

        # Heuristic: Online/Natural voices often can't render to file.
        voice_is_online = any(
            k in voice_label.lower() for k in ("online", "natural")
        )

        try:
            total = len(chunks)
            for i, chunk in enumerate(chunks, start=1):
                # Skip empties (shouldn't happen, but be safe).
                if not chunk or not chunk.strip():
                    self.after(0, self._ui_progress, i, total)
                    continue

                wav_path = tmp_dir / f"part_{i:03d}.wav"
                mp3_path = out_dir / f"Solina_Part_{i:03d}.mp3"

                self.after(0, self._ui_status,
                           f"Converting part {i}/{total}…")

                try:
                    # (a) WAV via direct SAPI5
                    self.tts.render_to_wav(
                        chunk, wav_path,
                        voice_id=voice_id,
                        rate=settings["rate"],
                        pitch=settings["pitch"],
                        volume=settings["volume"],
                    )
                    if not wav_path.exists() or wav_path.stat().st_size == 0:
                        raise RuntimeError(
                            f"SAPI produced no WAV for part {i:03d}"
                        )

                    # (b) WAV -> MP3 (also deletes the WAV on success)
                    wav_to_mp3(wav_path, mp3_path)

                    # (c) verify the MP3 on disk
                    if not mp3_path.exists() or mp3_path.stat().st_size == 0:
                        raise RuntimeError(
                            f"MP3 missing or empty: {mp3_path.name}"
                        )

                    ok += 1

                except Exception as e:
                    # Full traceback -> solina_log.txt; short message -> UI.
                    short = log_exc(f"Part {i:03d}", e)
                    hint = ""
                    if voice_is_online:
                        hint = (
                            f"  Voice '{voice_label}' looks like an Online/Natural "
                            "voice — SAPI file export often fails with these. "
                            "Try a Desktop voice (e.g. Microsoft David, Zira, "
                            "or Ava Desktop)."
                        )
                    errors.append(f"Part {i:03d}: {short}{hint}")
                finally:
                    # Best-effort cleanup of the temp WAV.
                    try:
                        if wav_path.exists():
                            wav_path.unlink()
                    except Exception:
                        pass

                # Progress update (safe from worker thread).
                self.after(0, self._ui_progress, i, total)

        finally:
            # Cleanup temp dir + COM apartment.
            try:
                for f in tmp_dir.iterdir():
                    try:
                        f.unlink()
                    except Exception:
                        pass
                tmp_dir.rmdir()
            except Exception:
                pass

            pythoncom.CoUninitialize()

            # Final completion dialog on the UI thread.
            self.after(0, self._on_export_finished,
                       ok, len(chunks), str(out_dir), errors)

    # ------------------------------------------------------------ UI plumbing
    def _ui_status(self, text: str):
        """Set the status bar text. Called via after() from the worker."""
        self.status_var.set(text)

    def _ui_progress(self, i: int, total: int):
        """Update the progress bar + status. Called via after() from worker."""
        self.progress.configure(value=i)
        self.status_var.set(f"Converting part {i}/{total}…")

    def _on_export_finished(self, ok: int, total: int, folder: str,
                            errors: list[str]):
        """Restore UI state and show the completion dialog (with Open folder)."""
        self._busy = False
        self.progress.configure(value=total)
        self.status_var.set(f"Done — {ok}/{total} files written.")
        self._show_completion(ok, total, folder, errors)

    def _show_completion(self, ok: int, total: int, folder: str,
                         errors: list[str]):
        """Custom completion popup with an 'Open folder' button."""
        win = tk.Toplevel(self)
        win.title("Solina — Export complete")
        win.configure(bg=BG_MAIN)
        win.geometry("520x360")
        win.transient(self)
        win.grab_set()

        head = tk.Label(
            win,
            text=f"Exported {ok} of {total} MP3 file(s).",
            bg=BG_MAIN, fg=FG_TEXT, font=("Segoe UI", 12, "bold"),
            justify="left",
        )
        head.pack(anchor="w", padx=16, pady=(16, 4))

        tk.Label(
            win, text=f"Folder:\n{folder}",
            bg=BG_MAIN, fg=FG_TEXT, justify="left", wraplength=480,
            font=("Segoe UI", 10),
        ).pack(anchor="w", padx=16)

        if errors:
            tk.Label(
                win,
                text="Some parts failed (see solina_log.txt for details):",
                bg=BG_MAIN, fg="#ff8080", font=("Segoe UI", 10, "bold"),
            ).pack(anchor="w", padx=16, pady=(12, 2))
            shown = "\n".join(errors[:6])
            if len(errors) > 6:
                shown += f"\n…and {len(errors) - 6} more."
            tk.Label(
                win, text=shown,
                bg=BG_MAIN, fg=FG_TEXT, justify="left", wraplength=480,
                font=("Consolas", 9),
            ).pack(anchor="w", padx=16)

        btns = tk.Frame(win, bg=BG_MAIN)
        btns.pack(side="bottom", fill="x", padx=16, pady=12)

        def open_folder():
            try:
                os.startfile(folder)  # noqa: S606 — Windows-only app
            except Exception as e:
                messagebox.showerror("Solina", f"Could not open folder:\n{e}")

        tb.Button(btns, text="Open folder", bootstyle="info",
                  command=open_folder).pack(side="left")
        tb.Button(btns, text="Close", bootstyle="secondary",
                  command=win.destroy).pack(side="right")

    # ------------------------------------------------------------------ close
    def _on_close(self):
        """Stop any running speech and destroy the window cleanly."""
        try:
            self.tts.engine.stop()
        except Exception:
            pass
        self.destroy()

NameError: name 'tb' is not defined

## WAV → MP3 helper

In [7]:
# pydub wraps ffmpeg. We convert the temp WAV produced by SAPI5 into an MP3
# at the user's chosen output folder.

def wav_to_mp3(wav_path: Path, mp3_path: Path) -> None:
    """
    Convert a WAV file to MP3 using pydub/ffmpeg.

    Raises a clear error if ffmpeg is missing so the GUI can show a helpful message.
    """
    try:
        audio = AudioSegment.from_wav(str(wav_path))
    except Exception as e:
        raise RuntimeError(f"Could not read WAV file {wav_path}: {e}") from e

    try:
        audio.export(str(mp3_path), format="mp3")
    except FileNotFoundError as e:
        raise RuntimeError(
            "ffmpeg was not found on PATH. Install it (see the setup cell) "
            "and restart Jupyter."
        ) from e

## Main GUI class

In [18]:
# The complete Tkinter/ttkbootstrap UI.
# Layout: toolbar row -> text editor (left) + controls (right) -> status bar.
# Long-running work (preview, export) runs on background threads and posts
# progress back to the UI through a queue polled with Tk's `after()`.

# ---- GUI imports (needed by SolinaApp; safe to repeat) ----
import os
import threading
import tkinter as tk
from tkinter import filedialog, messagebox
from pathlib import Path

import ttkbootstrap as tb                  # "tb" is the alias used in the class
from ttkbootstrap.constants import *       # BOTH, LEFT, X, Y, etc.

ACCENT   = "#4ea1ff"      # single accent colour
BG_MAIN  = "#2b2b2b"      # window background
BG_DARK  = "#1e1e1e"      # text-area background
FG_TEXT  = "#e6e6e6"      # light-grey text


class SolinaApp(tb.Window):
    """Solina main window."""

    # --------------------------------------------------------------- lifecycle
    def __init__(self):
        super().__init__(themename="darkly")   # darkly is close to #2b2b2b
        self.title("Solina — Text to Speech")
        self.geometry("1180x800")
        self.minsize(960, 640)
        self.configure(background=BG_MAIN)

        # Core objects ------------------------------------------------------
        self.tts = TTSEngine()
        self._msg_q: queue.Queue = queue.Queue()      # worker -> UI messages
        self._busy = False

        # Tk variables ------------------------------------------------------
        self.voice_choice = tk.StringVar()
        self.rate_var     = tk.IntVar(value=180)
        self.pitch_var    = tk.IntVar(value=0)
        self.volume_var   = tk.DoubleVar(value=1.0)

        self.split_mode   = tk.StringVar(value="chars")
        self.char_count   = tk.StringVar(value="3000")
        self.word_count   = tk.StringVar(value="500")
        self.keywords     = tk.StringVar(value="")
        self.strip_kw     = tk.BooleanVar(value=True)

        self.output_dir   = tk.StringVar(value=str(Path.home() / "Solina_Output"))
        self.status_var   = tk.StringVar(value="Ready")
        self.file_var     = tk.StringVar(value="(no file loaded)")

        self._voice_map: dict[str, str] = {}   # display label -> SAPI voice id

        # Build & populate --------------------------------------------------
        self._build_ui()
        self._populate_voices()
        self.protocol("WM_DELETE_WINDOW", self._on_close)

    # ------------------------------------------------------------------ build
    def _build_ui(self):
        """Create every widget in the window exactly once."""
        style = self.style
        # Custom style names keep our dark palette even if the user changes theme.
        style.configure("Solina.TFrame", background=BG_MAIN)
        style.configure("Solina.TLabel", background=BG_MAIN, foreground=FG_TEXT)
        style.configure("Solina.TLabelframe", background=BG_MAIN,
                        foreground=FG_TEXT, bordercolor="#3c3c3c")
        style.configure("Solina.TLabelframe.Label", background=BG_MAIN, foreground=FG_TEXT)
        style.configure("Solina.TRadiobutton", background=BG_MAIN, foreground=FG_TEXT)
        style.configure("Solina.TCheckbutton", background=BG_MAIN, foreground=FG_TEXT)
        style.configure("Solina.TEntry", fieldbackground=BG_DARK, foreground=FG_TEXT)
        style.configure("Accent.TButton", background=ACCENT, foreground="#0b0b0b")

        root = tb.Frame(self, padding=14, style="Solina.TFrame")
        root.pack(fill=BOTH, expand=True)
        root.columnconfigure(0, weight=1)
        root.columnconfigure(1, weight=0)
        root.rowconfigure(1, weight=1)

        # ---- Row 0: toolbar ------------------------------------------------
        bar = tb.Frame(root, style="Solina.TFrame")
        bar.grid(row=0, column=0, columnspan=2, sticky="ew", pady=(0, 12))

        tb.Button(bar, text="Import file…", bootstyle="secondary",
                  command=self._on_import).pack(side=LEFT)
        tb.Label(bar, textvariable=self.file_var, style="Solina.TLabel"
                 ).pack(side=LEFT, padx=(10, 0))

        tb.Button(bar, text="Browse…", bootstyle="secondary",
                  command=self._on_browse_output).pack(side=RIGHT)
        tb.Label(bar, textvariable=self.output_dir, style="Solina.TLabel"
                 ).pack(side=RIGHT, padx=(0, 10))
        tb.Label(bar, text="Output folder:", style="Solina.TLabel").pack(side=RIGHT)

        # ---- Row 1 left: text editor ---------------------------------------
        left = tb.Frame(root, style="Solina.TFrame")
        left.grid(row=1, column=0, sticky="nsew", padx=(0, 14))
        left.rowconfigure(1, weight=1)
        left.columnconfigure(0, weight=1)

        tb.Label(left, text="Text content", style="Solina.TLabel",
                 font=("Segoe UI", 12, "bold")).grid(row=0, column=0, sticky=W)

        text_wrap = tk.Frame(left, bg=BG_DARK)
        text_wrap.grid(row=1, column=0, sticky="nsew", pady=(6, 0))
        text_wrap.rowconfigure(0, weight=1)
        text_wrap.columnconfigure(0, weight=1)

        self.text = tk.Text(
            text_wrap, wrap="word", undo=True,
            bg=BG_DARK, fg=FG_TEXT, insertbackground=FG_TEXT,
            selectbackground=ACCENT, selectforeground="#000000",
            relief="flat", padx=12, pady=12, font=("Consolas", 11),
        )
        self.text.grid(row=0, column=0, sticky="nsew")

        sb = tb.Scrollbar(text_wrap, orient="vertical", command=self.text.yview)
        sb.grid(row=0, column=1, sticky="ns")
        self.text.configure(yscrollcommand=sb.set)

        # ---- Row 1 right: controls -----------------------------------------
        right = tb.Frame(root, style="Solina.TFrame")
        right.grid(row=1, column=1, sticky="ns")

        # Voice + audio
        audio_lf = tb.Labelframe(right, text="Voice & audio", padding=12,
                                 style="Solina.TLabelframe")
        audio_lf.pack(fill=X, pady=(0, 10))

        tb.Label(audio_lf, text="Voice", style="Solina.TLabel").pack(anchor=W)
        self.voice_combo = tb.Combobox(audio_lf, textvariable=self.voice_choice,
                                       state="readonly", width=38)
        self.voice_combo.pack(fill=X, pady=(2, 10))

        self._slider(audio_lf, "Rate",   self.rate_var,   50, 400, 1)
        self._slider(audio_lf, "Pitch",  self.pitch_var, -50,  50, 1)
        self._slider(audio_lf, "Volume", self.volume_var,  0.0, 1.0, 0.01)

        tb.Button(audio_lf, text="Test voice", bootstyle="info",
                  command=self._on_test_voice).pack(fill=X, pady=(10, 0))

        # Splitting
        split_lf = tb.Labelframe(right, text="Text splitting", padding=12,
                                 style="Solina.TLabelframe")
        split_lf.pack(fill=X)

        def radio(value, label):
            tb.Radiobutton(split_lf, text=label, value=value,
                           variable=self.split_mode,
                           style="Solina.TRadiobutton").pack(anchor=W)

        radio("chars",    "By block size — characters")
        row = tb.Frame(split_lf, style="Solina.TFrame"); row.pack(fill=X, padx=(22, 0))
        tb.Entry(row, textvariable=self.char_count, width=8).pack(side=LEFT)
        tb.Label(row, text="characters", style="Solina.TLabel").pack(side=LEFT, padx=6)

        radio("words",    "By block size — words")
        row = tb.Frame(split_lf, style="Solina.TFrame"); row.pack(fill=X, padx=(22, 0))
        tb.Entry(row, textvariable=self.word_count, width=8).pack(side=LEFT)
        tb.Label(row, text="words", style="Solina.TLabel").pack(side=LEFT, padx=6)

        radio("keywords", "By keyword list")
        row = tb.Frame(split_lf, style="Solina.TFrame"); row.pack(fill=X, padx=(22, 0))
        tb.Entry(row, textvariable=self.keywords, width=26).pack(side=LEFT)
        tb.Checkbutton(split_lf, text="Strip keyword from output",
                       variable=self.strip_kw,
                       style="Solina.TCheckbutton").pack(anchor=W, padx=(22, 0))

        radio("blank",    "By two consecutive empty lines")
        radio("caps",     "By lines that are ALL CAPS (headings)")

        # Export
        tb.Button(right, text="Convert to MP3", bootstyle="success",
                  command=self._on_convert).pack(fill=X, pady=(12, 0))

        # ---- Row 2: status + progress --------------------------------------
        bottom = tb.Frame(root, style="Solina.TFrame")
        bottom.grid(row=2, column=0, columnspan=2, sticky="ew", pady=(12, 0))
        bottom.columnconfigure(0, weight=1)

        self.progress = tb.Progressbar(bottom, mode="determinate",
                                       bootstyle="info-striped")
        self.progress.grid(row=0, column=0, sticky="ew")
        tb.Label(bottom, textvariable=self.status_var, style="Solina.TLabel"
                 ).grid(row=1, column=0, sticky=W, pady=(6, 0))

    def _slider(self, parent, label, var, lo, hi, step):
        """Helper: build a labelled horizontal slider."""
        row = tb.Frame(parent, style="Solina.TFrame")
        row.pack(fill=X, pady=(2, 0))
        tb.Label(row, text=label, width=7, style="Solina.TLabel").pack(side=LEFT)
        tb.Scale(row, from_=lo, to=hi, variable=var, orient=HORIZONTAL
                 ).pack(side=LEFT, fill=X, expand=True)

    # ------------------------------------------------------------------ voices
    def _populate_voices(self):
        """Fill the voice dropdown with installed SAPI5 voices."""
        voices = self.tts.list_voices()
        self._voice_map.clear()
        labels = []
        for v in voices:
            langs = ", ".join(v["langs"]) if v["langs"] else "unknown"
            label = f'{v["name"]}  ({langs})'
            labels.append(label)
            self._voice_map[label] = v["id"]
        self.voice_combo["values"] = labels
        if labels:
            self.voice_combo.current(0)

    # --------------------------------------------------------------- callbacks
    def _on_import(self):
        """Load .txt/.pdf/.epub into the text box."""
        path = filedialog.askopenfilename(
            title="Choose a source file",
            filetypes=[("Supported files", "*.txt *.pdf *.epub"),
                       ("Text", "*.txt"), ("PDF", "*.pdf"), ("EPUB", "*.epub")],
        )
        if not path:
            return
        try:
            content = extract_text(path)
        except Exception as e:
            messagebox.showerror("Solina", f"Failed to read file:\n{e}")
            return
        self.text.delete("1.0", END)
        self.text.insert("1.0", content)
        self.file_var.set(Path(path).name)

    def _on_browse_output(self):
        """Pick the output folder for exported MP3s."""
        folder = filedialog.askdirectory(title="Choose output folder")
        if folder:
            self.output_dir.set(folder)

    def _on_test_voice(self):
        """Speak a sample using the current settings (no file export)."""
        if self._busy:
            return
        voice_id = self._voice_map.get(self.voice_choice.get())
        rate, pitch = self.rate_var.get(), self.pitch_var.get()
        vol = self.volume_var.get()
        sample = "Hello, this is Solina. This is how your audiobook will sound."
        self.status_var.set("Previewing voice…")

        def worker():
            try:
                self.tts.preview(sample, voice_id, rate, pitch, vol)
                self._msg_q.put(("status", "Preview finished."))
            except Exception as e:
                self._msg_q.put(("error", f"Preview failed: {e}"))

        threading.Thread(target=worker, daemon=True).start()

    def _make_chunks(self, text: str) -> list[str]:
        """Apply the currently selected splitting strategy."""
        mode = self.split_mode.get()
        if mode == "chars":
            return split_by_chars(text, int(self.char_count.get() or 3000))
        if mode == "words":
            return split_by_words(text, int(self.word_count.get() or 500))
        if mode == "keywords":
            return split_by_keywords(text, self.keywords.get().split(","),
                                     strip=self.strip_kw.get())
        if mode == "blank":
            return split_by_blank_lines(text)
        if mode == "caps":
            return split_by_capital_lines(text)
        return [text]

    def _on_convert(self):
        """Validate inputs, confirm the file count, and start the export thread."""
        if self._busy:
            return

        text = self.text.get("1.0", END).strip()
        if not text:
            messagebox.showwarning("Solina", "There is no text to convert.")
            return

        # --- Build chunks; convert int-parsing errors into a friendly popup ---
        try:
            chunks = self._make_chunks(text)
        except ValueError as e:
            messagebox.showerror("Solina", f"Invalid split parameter: {e}")
            return

        if not chunks:
            messagebox.showwarning("Solina", "The split settings produced 0 chunks.")
            return

        # --- Approximate file count confirmation -----------------------------
        proceed = messagebox.askyesno(
            "Solina",
            f"Approximate number of files: {len(chunks)}\n\nProceed with export?",
        )
        if not proceed:
            return

        out_dir = Path(self.output_dir.get()).expanduser()
        try:
            out_dir.mkdir(parents=True, exist_ok=True)
        except Exception as e:
            messagebox.showerror("Solina", f"Could not create output folder:\n{e}")
            return

        # --- Snapshot Tk variables (workers must not read Tk state) ----------
        settings = {
            "voice_id": self._voice_map.get(self.voice_choice.get()),
            "rate":     self.rate_var.get(),
            "pitch":    self.pitch_var.get(),
            "volume":   self.volume_var.get(),
        }

        self._busy = True
        self.progress.configure(maximum=len(chunks), value=0)
        self.status_var.set(f"0 / {len(chunks)}")

        threading.Thread(
            target=self._convert_worker,
            args=(chunks, out_dir, settings),
            daemon=True,
        ).start()
        self.after(100, self._drain_queue)

    # ----------------------------------------------------------------- worker
    def _convert_worker(self, chunks, out_dir: Path, settings: dict):
        """Render each chunk to WAV, convert to MP3, report progress via queue."""
        tmp_dir = Path(tempfile.mkdtemp(prefix="solina_"))
        ok = 0
        errors: list[str] = []

        for i, chunk in enumerate(chunks, start=1):
            wav_path = tmp_dir / f"part_{i:03d}.wav"
            mp3_path = out_dir / f"Solina_Part_{i:03d}.mp3"
            try:
                self.tts.render_to_wav(
                    chunk, wav_path,
                    voice_id=settings["voice_id"],
                    rate=settings["rate"],
                    pitch=settings["pitch"],
                    volume=settings["volume"],
                )
                wav_to_mp3(wav_path, mp3_path)
                ok += 1
            except Exception as e:
                errors.append(f"Part {i:03d}: {e}")
            finally:
                # Best-effort cleanup of the intermediate WAV.
                try:
                    wav_path.unlink(missing_ok=True)
                except Exception:
                    pass
            self._msg_q.put(("progress", i, len(chunks)))

        self._msg_q.put(("done", ok, len(chunks), str(out_dir), errors))

    # ---------------------------------------------------------- queue poller
    def _drain_queue(self):
        """Process messages posted by the worker thread; reschedule while busy."""
        try:
            while True:
                msg = self._msg_q.get_nowait()
                kind = msg[0]

                if kind == "progress":
                    _, i, total = msg
                    self.progress.configure(value=i)
                    self.status_var.set(f"Rendering {i} / {total}…")

                elif kind == "status":
                    self.status_var.set(msg[1])

                elif kind == "error":
                    self.status_var.set(msg[1])

                elif kind == "done":
                    _, ok, total, folder, errors = msg
                    self._busy = False
                    self.progress.configure(value=total)
                    self.status_var.set(f"Done — {ok}/{total} files written.")
                    summary = f"Exported {ok} of {total} MP3 files to:\n{folder}"
                    if errors:
                        summary += "\n\nSome parts failed:\n" + "\n".join(errors[:8])
                        if len(errors) > 8:
                            summary += f"\n…and {len(errors) - 8} more."
                        messagebox.showwarning("Solina — Export complete", summary)
                    else:
                        messagebox.showinfo("Solina — Export complete", summary)
                    return
        except queue.Empty:
            pass

        if self._busy:
            self.after(100, self._drain_queue)

    # ------------------------------------------------------------------ close
    def _on_close(self):
        """Stop any running speech and destroy the window cleanly."""
        try:
            self.tts.engine.stop()
        except Exception:
            pass
        self.destroy()

## Launch the app

In [ ]:
# Standard Python entry point. In Jupyter this launches the Tk main loop and
# blocks the cell until you close the Solina window.

if __name__ == "__main__":
    app = SolinaApp()
    app.mainloop()

NameError: name 'SolinaApp' is not defined